# Lab 8: Parsing

Expand the grammar of the non-probabilistic chart parser example so that it covers the sentence *"Lazy cats play with mice."*, parse it with a `BottomUpChartParser`, a `BottomUpLeftCornerChartParser` and a `LeftCornerChartParser`, and compare the trees, the number of edges and the edges explored by each parser.

In [1]:
import sys
from pathlib import Path

# Colab: install packages, mount Google Drive and use the Colab Notebooks/ihlt folder as root.
# Local: notebooks live in labs/, shared resources in the project root.
# This is just some setup for file organization
if "google.colab" in sys.modules:
    %pip install -q nltk pandas
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/Colab Notebooks/ihlt")
else:
    ROOT = Path.cwd().parent

# Notebook libs
import nltk
import pandas as pd
from nltk import CFG
from nltk.parse.chart import BottomUpChartParser, BottomUpLeftCornerChartParser, LeftCornerChartParser

In [2]:
# Grammar of the example in the slides.
example_grammar = CFG.fromstring("""
    NP -> NNS | JJ NNS | NP CC NP
    NNS -> "cats" | "dogs" | "mice" | NNS CC NNS
    JJ -> "big" | "small"
    CC -> "and" | "or"
""")

# Expanded grammar: the rules of the example are kept, and the following ones are added:
# - S, the new start symbol: a sentence is a NP subject, a VP and the final punctuation.
# - VP: a verb, optionally followed by a PP ("play" / "play with mice").
# - PP: a preposition followed by a NP ("with mice").
# - The new words: "lazy" (with and without capital letter, as it starts the sentence), "play", "with" and ".".
grammar = CFG.fromstring("""
    S -> NP VP PUNCT
    VP -> VBP | VBP PP
    PP -> IN NP
    NP -> NNS | JJ NNS | NP CC NP
    NNS -> "cats" | "dogs" | "mice" | NNS CC NNS
    JJ -> "big" | "small" | "lazy" | "Lazy"
    CC -> "and" | "or"
    VBP -> "play"
    IN -> "with"
    PUNCT -> "."
""")

sentence = ["Lazy", "cats", "play", "with", "mice", "."]
print(f"Start symbol: {grammar.start()}")
print(f"Productions: {len(example_grammar.productions())} in the example, {len(grammar.productions())} in the expanded grammar")

# The expanded grammar still parses the example of the slides (as a NP).
example = ["small", "cats", "and", "mice"]
np_grammar = CFG(nltk.Nonterminal("NP"), grammar.productions())
print(f"Trees of {example} with the expanded grammar (start NP): {len(list(BottomUpChartParser(np_grammar).parse(example)))}")

Start symbol: S
Productions: 11 in the example, 20 in the expanded grammar
Trees of ['small', 'cats', 'and', 'mice'] with the expanded grammar (start NP): 2


In [3]:
# Parse the sentence with the three parsers.
parsers = {
    "BottomUpChartParser": BottomUpChartParser(grammar),
    "BottomUpLeftCornerChartParser": BottomUpLeftCornerChartParser(grammar),
    "LeftCornerChartParser": LeftCornerChartParser(grammar),
}
charts = {name: parser.chart_parse(sentence) for name, parser in parsers.items()}
trees = {name: list(chart.parses(grammar.start())) for name, chart in charts.items()}

for name, chart in charts.items():
    print(f"=== {name}: {chart.num_edges()} edges, {len(trees[name])} tree(s)\n")
    for tree in trees[name]:
        tree.pretty_print()

=== BottomUpChartParser: 55 edges, 1 tree(s)

               S                      
       ________|___________________    
      |             VP             |  
      |         ____|____          |   
      |        |         PP        |  
      |        |     ____|___      |   
      NP       |    |        NP    |  
  ____|___     |    |        |     |   
 JJ      NNS  VBP   IN      NNS  PUNCT
 |        |    |    |        |     |   
Lazy     cats play with     mice   .  

=== BottomUpLeftCornerChartParser: 35 edges, 1 tree(s)

               S                      
       ________|___________________    
      |             VP             |  
      |         ____|____          |   
      |        |         PP        |  
      |        |     ____|___      |   
      NP       |    |        NP    |  
  ____|___     |    |        |     |   
 JJ      NNS  VBP   IN      NNS  PUNCT
 |        |    |    |        |     |   
Lazy     cats play with     mice   .  

=== LeftCornerChartParser: 2

In [4]:
# List of explored edges of each parser, in the order they were added to the chart.
# [i:j] is the span of the edge, and * the dot: what is on its left has already been found.
for name, chart in charts.items():
    print(f"=== {name} ({chart.num_edges()} edges)")
    for n, edge in enumerate(chart.edges(), 1):
        print(f"{n:>3}  {edge}")
    print()

=== BottomUpChartParser (55 edges)
  1  [0:1] 'Lazy'
  2  [1:2] 'cats'
  3  [2:3] 'play'
  4  [3:4] 'with'
  5  [4:5] 'mice'
  6  [5:6] '.'
  7  [0:0] JJ -> * 'Lazy'
  8  [0:1] JJ -> 'Lazy' *
  9  [0:0] NP -> * JJ NNS
 10  [0:1] NP -> JJ * NNS
 11  [1:1] NNS -> * 'cats'
 12  [1:2] NNS -> 'cats' *
 13  [1:1] NP -> * NNS
 14  [1:1] NNS -> * NNS CC NNS
 15  [0:2] NP -> JJ NNS *
 16  [1:2] NP -> NNS *
 17  [1:2] NNS -> NNS * CC NNS
 18  [1:1] S  -> * NP VP PUNCT
 19  [1:1] NP -> * NP CC NP
 20  [1:2] S  -> NP * VP PUNCT
 21  [1:2] NP -> NP * CC NP
 22  [0:0] S  -> * NP VP PUNCT
 23  [0:0] NP -> * NP CC NP
 24  [0:2] S  -> NP * VP PUNCT
 25  [0:2] NP -> NP * CC NP
 26  [2:2] VBP -> * 'play'
 27  [2:3] VBP -> 'play' *
 28  [2:2] VP -> * VBP
 29  [2:2] VP -> * VBP PP
 30  [2:3] VP -> VBP *
 31  [2:3] VP -> VBP * PP
 32  [1:3] S  -> NP VP * PUNCT
 33  [0:3] S  -> NP VP * PUNCT
 34  [3:3] IN -> * 'with'
 35  [3:4] IN -> 'with' *
 36  [3:3] PP -> * IN NP
 37  [3:4] PP -> IN * NP
 38  [4:4] NNS -

In [5]:
# Summary: edges by type.
# - leaf: the words of the sentence.
# - zero-width: predicted edges that have not consumed any word yet ([i:i] A -> * ...).
# - incomplete: edges with something left after the dot.
# - complete: edges whose right-hand side has been fully found.
def edge_type(edge):
    if isinstance(edge, nltk.parse.chart.LeafEdge):
        return "leaf"
    if edge.length() == 0:
        return "zero-width"
    return "complete" if edge.is_complete() else "incomplete"

summary = pd.DataFrame({name: pd.Series([edge_type(e) for e in chart.edges()]).value_counts()
                        for name, chart in charts.items()}).T.fillna(0).astype(int)
summary = summary[["leaf", "zero-width", "incomplete", "complete"]]
summary["total"] = summary.sum(axis=1)
summary

,leaf,zero-width,incomplete,complete,total
BottomUpChartParser,6,20,15,14,55
BottomUpLeftCornerChartParser,6,0,15,14,35
LeftCornerChartParser,6,0,7,14,27


In [6]:
# Which edges does each parser filter out with respect to the previous one?
edges = {name: {str(e) for e in chart.edges()} for name, chart in charts.items()}
bu, bulc, lc = edges.values()

print(f"Edges of BottomUpLeftCornerChartParser not in BottomUpChartParser: {len(bulc - bu)}")
print(f"Edges of LeftCornerChartParser not in BottomUpLeftCornerChartParser: {len(lc - bulc)}\n")

print(f"=== Filtered out by BottomUpLeftCornerChartParser (in BottomUp, not in BottomUpLeftCorner): {len(bu - bulc)}")
for e in sorted(bu - bulc):
    print("   ", e)

print(f"\n=== Filtered out by LeftCornerChartParser (in BottomUpLeftCorner, not in LeftCorner): {len(bulc - lc)}")
for e in sorted(bulc - lc):
    print("   ", e)

Edges of BottomUpLeftCornerChartParser not in BottomUpChartParser: 0
Edges of LeftCornerChartParser not in BottomUpLeftCornerChartParser: 0

=== Filtered out by BottomUpLeftCornerChartParser (in BottomUp, not in BottomUpLeftCorner): 20
    [0:0] JJ -> * 'Lazy'
    [0:0] NP -> * JJ NNS
    [0:0] NP -> * NP CC NP
    [0:0] S  -> * NP VP PUNCT
    [1:1] NNS -> * 'cats'
    [1:1] NNS -> * NNS CC NNS
    [1:1] NP -> * NNS
    [1:1] NP -> * NP CC NP
    [1:1] S  -> * NP VP PUNCT
    [2:2] VBP -> * 'play'
    [2:2] VP -> * VBP
    [2:2] VP -> * VBP PP
    [3:3] IN -> * 'with'
    [3:3] PP -> * IN NP
    [4:4] NNS -> * 'mice'
    [4:4] NNS -> * NNS CC NNS
    [4:4] NP -> * NNS
    [4:4] NP -> * NP CC NP
    [4:4] S  -> * NP VP PUNCT
    [5:5] PUNCT -> * '.'

=== Filtered out by LeftCornerChartParser (in BottomUpLeftCorner, not in LeftCorner): 8
    [0:2] NP -> NP * CC NP
    [0:3] S  -> NP VP * PUNCT
    [1:2] NNS -> NNS * CC NNS
    [1:2] NP -> NP * CC NP
    [1:3] S  -> NP VP * PUNCT
    [4:

In [7]:
# Why are these edges filtered out by the LeftCornerChartParser?
# Its filter only adds an edge if the next word of the sentence can start the next expected symbol.
# Words that can start each symbol (the "left corner" words) according to the grammar:
def first_words(symbol, seen=()):
    words = set()
    for prod in grammar.productions(lhs=symbol):
        first = prod.rhs()[0]
        if isinstance(first, str):
            words.add(first)
        elif first not in seen:
            words |= first_words(first, seen + (symbol,))
    return words

for symbol in ["CC", "PUNCT", "VP", "PP", "NP"]:
    print(f"{symbol:<6} can start with: {sorted(first_words(nltk.Nonterminal(symbol)))}")

print()
for e in sorted(charts["BottomUpLeftCornerChartParser"].edges(), key=str):
    if str(e) in bulc - lc:
        expected = e.nextsym()
        next_word = sentence[e.end()] if e.end() < len(sentence) else "(end of sentence)"
        print(f"{str(e):<32} expects {str(expected):<6} next word: {next_word!r}")

CC     can start with: ['and', 'or']
PUNCT  can start with: ['.']
VP     can start with: ['play']
PP     can start with: ['with']
NP     can start with: ['Lazy', 'big', 'cats', 'dogs', 'lazy', 'mice', 'small']

[0:2] NP -> NP * CC NP           expects CC     next word: 'play'
[0:3] S  -> NP VP * PUNCT        expects PUNCT  next word: 'with'
[1:2] NNS -> NNS * CC NNS        expects CC     next word: 'play'
[1:2] NP -> NP * CC NP           expects CC     next word: 'play'
[1:3] S  -> NP VP * PUNCT        expects PUNCT  next word: 'with'
[4:5] NNS -> NNS * CC NNS        expects CC     next word: '.'
[4:5] NP -> NP * CC NP           expects CC     next word: '.'
[4:5] S  -> NP * VP PUNCT        expects VP     next word: '.'


## Questions

### Grammar and results

The grammar of the example is expanded with a start symbol `S -> NP VP PUNCT`, a `VP` (verb with an optional `PP`), a `PP` (preposition + `NP`) and the new words. All the rules of the example are kept, so the expanded grammar still gives the 2 trees of *"small cats and mice"*.

The three parsers find the same and only tree: `(S (NP (JJ Lazy) (NNS cats)) (VP (VBP play) (PP (IN with) (NP (NNS mice)))) (PUNCT .))`. The difference is in the number of edges they create:

| Parser | Leaf | Zero-width | Incomplete | Complete | Total |
|---|---|---|---|---|---|
| BottomUpChartParser | 6 | 20 | 15 | 14 | 55 |
| BottomUpLeftCornerChartParser | 6 | 0 | 15 | 14 | 35 |
| LeftCornerChartParser | 6 | 0 | 7 | 14 | 27 |

### Which parser is the most efficient for parsing the sentence?

**The LeftCornerChartParser**: it builds the same tree with 27 edges, against 35 for the BottomUpLeftCornerChartParser and 55 for the BottomUpChartParser (about half). The three parsers create the same 14 complete edges; the difference is in the predicted edges, which the more efficient parsers avoid creating. Each parser only removes edges with respect to the previous one: no parser creates an edge that the previous one does not create.

### Which edges are filtered out by each parser and why?

**BottomUpChartParser → BottomUpLeftCornerChartParser: the 20 zero-width edges.** The BottomUpChartParser uses the bottom-up predict rule: for every complete edge of a symbol `B` at position `i`, it adds an empty edge `[i:i] A -> * B ...` for every rule whose right-hand side starts with `B`, e.g. `[0:0] JJ -> * 'Lazy'`, `[0:0] NP -> * JJ NNS` or `[2:2] VP -> * VBP PP`. Only later the fundamental rule combines them with the complete edge and moves the dot. The BottomUpLeftCornerChartParser does both steps at once (predict + combine): it directly adds `[0:1] NP -> JJ * NNS`, with the first symbol already found. The zero-width edges are just an intermediate step, so this parser saves all 20 of them without losing any information.

**BottomUpLeftCornerChartParser → LeftCornerChartParser: 8 incomplete edges.** The LeftCornerChartParser adds a filter that looks at the next word of the sentence: an incomplete edge is only added if the next word can be the first word (the left corner) of the next symbol it expects. The 8 filtered edges are the ones that can never be completed in this sentence:

- `NP -> NP * CC NP` and `NNS -> NNS * CC NNS` (after *cats* and after *mice*): they expect a `CC`, which can only start with *and* or *or*, but the next word is *play* or *.*. There is no coordination in this sentence.
- `S -> NP VP * PUNCT` over *"Lazy cats play"* and *"cats play"*: they expect a `PUNCT` (*.*), but the next word is *with*.
- `S -> NP * VP PUNCT` after *mice*: it expects a `VP`, which can only start with *play*, but the next word is *.*.

The LeftCornerChartParser still creates some useless edges, because its filter only looks at the next word, not at the whole sentence. For example it builds `[1:2] NP -> NNS *` (*cats* alone as a NP), `[1:2] S -> NP * VP PUNCT` and finally the complete edge `[1:6] S -> NP VP PUNCT *` for *"cats play with mice ."*, which does not start at the first word and so is not a parse of the sentence. The difference is also small here because the sentence and the grammar are small; with longer sentences and larger grammars, filtering the predictions saves much more work.